<a href="https://colab.research.google.com/github/shulruferor-sudo/Advance-Data-Science/blob/main/projects/Bureau%20of%20Statistics/Assignment_Solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shulruferor-sudo/Advance-Data-Science/blob/main/projects/Bureau%20of%20Statistics/Assignment_Solution.ipynb)


# Occupational Classification: Survey Data and Model Evaluation

This notebook examines survey data and metadata, tests field-specific preparation decisions, and evaluates occupational classification models. Each transformation and modeling decision is tied to observable evidence and assessed against its alternatives.


In [ ]:
# Load the tools needed to trace survey answers from raw data to evaluated occupation predictions.
import io
import hashlib
import urllib.request
import re
import json
import warnings
from pathlib import Path
from html import escape
import importlib
import subprocess
import sys
from urllib.request import urlopen

warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, Markdown, display

pd.set_option("display.html.use_mathjax", False)
pd.set_option("display.max_columns", None)


def custom_style(dataframe, *, grouped_id_column=None):
    """
    Apply the notebook’s table style and wrap grouped record IDs.
    """
    styled = dataframe.rename_axis(index=None, columns=None).style.format(precision=2, na_rep="—")
    if grouped_id_column is not None:
        styled = styled.format(escape="html", na_rep="—").set_properties(subset=[grouped_id_column], **{"max-width": "32em", "white-space": "normal"})
    elif "ID" in dataframe.columns:
        styled = styled.format({"ID": lambda value: f"{int(value)}" if pd.notna(value) else "—"})
    if isinstance(dataframe.index, pd.RangeIndex) and dataframe.index.equals(pd.RangeIndex(len(dataframe))):
        styled = styled.hide(axis="index")

    header_properties = [("background-color", "#1e293b"), ("color", "#ffffff"), ("font-family", "sans-serif"), ("font-size", "13px"), ("padding", "10px 15px"), ("text-align", "left")]
    cell_properties = [("font-family", "sans-serif"), ("font-size", "12px"), ("padding", "8px 15px"), ("border-bottom", "1px solid #e2e8f0")]
    styled = styled.set_table_styles([{"selector": "th", "props": header_properties}, {"selector": "td", "props": cell_properties}, {"selector": "tr:hover", "props": [("background-color", "#f1f5f9")]}])
    return styled.set_properties(**{"background-color": "#f8fafc"})


def style_report_text(text, role="body"):
    """
    Render report text using the style assigned to its role.
    """
    field_style = ("h3", "color: #1e293b; font-size: 18px; font-weight: 700; border-top: 1px solid #cbd5e1; padding-top: 12px; margin: 20px 0 8px")
    section_style = ("h4", "color: #0f766e; font-size: 14px; font-weight: 700; margin: 16px 0 4px")
    body_style = ("p", "color: #334155; font-size: 13px; margin: 4px 0")
    summary_style = ("p", "color: #475569; font-size: 13px; font-weight: 600; margin: 4px 0 8px")
    empty_style = ("p", "color: #64748b; font-size: 13px; font-style: italic; margin: 4px 0 8px")
    styles = {"field": field_style, "section": section_style, "body": body_style, "summary": summary_style, "empty": empty_style}
    tag, style = styles[role]
    return HTML(f"<{tag} style='font-family: sans-serif; {style}'>{escape(text)}</{tag}>")


pd.DataFrame._repr_html_ = lambda self: custom_style(self)._repr_html_()

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, GroupKFold, cross_val_predict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.dummy import DummyClassifier

from sklearn.preprocessing import Normalizer
from sklearn.decomposition import TruncatedSVD
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted

from sklearn.metrics import accuracy_score, f1_score, classification_report

SEED = 42

In [ ]:
def transform_to_tfidf(column, feature_fit_rows, norm=None, smooth_idf=False, sublinear_tf=False, use_idf=True):
    """
    Fit TF-IDF on selected records and return one pandas column per learned token.

    Parameters
    ----------
    column : pandas.Series
        A source column with one string per record. Use empty strings for
        missing values so those records produce rows of zeros. The output
        retains the Series index and record order.

    feature_fit_rows : sequence, pandas.Index, slice, or boolean mask
        Selects the records used to learn the vocabulary and IDF weights.
        Selection uses column.loc. Integer labels refer to the Series index.
        A boolean Series must align with that index. An unindexed boolean
        mask must have one entry per record. The selected records must
        collectively contain at least one token.

    norm : {None, "l1", "l2"}, default=None
        Controls normalization after weighting. None preserves the computed
        weights. "l1" scales each nonzero row so its absolute weights sum to
        1. "l2" scales each nonzero row to unit Euclidean length. Rows of
        zeros stay zero.

    smooth_idf : bool, default=False
        Controls inverse document frequency smoothing. False uses
        log(n / df) + 1. True uses log((n + 1) / (df + 1)) + 1.
        Here n is the number of fitting records and df is the number of
        those records containing the token. This option has no effect when
        use_idf is False.

    sublinear_tf : bool, default=False
        Controls how repeated tokens count within a record. False uses the
        occurrence count. True replaces each positive count with
        1 + log(count), reducing the weight of repeated occurrences.

    use_idf : bool, default=True
        Controls whether inverse document frequency weights multiply token
        frequencies. When False, sublinear_tf and norm still apply.

    Returns
    -------
    pandas.DataFrame
        Sparse numeric features with one row per input record and one
        column per learned token. For a named Series, each feature name is
        the source column name, two underscores, and the token, such as
        ShemAvoda__nvidia. An unnamed Series uses the tokens as names.

    Notes
    -----
    The vectorizer lowercases input and includes single-character word
    tokens. Tokens absent from the fitting vocabulary contribute no features.
    """
    column = column.astype("string").fillna("").str.lower().map(lambda value: re.sub(r"[^a-zא-ת ]", "", value))

    vectorizer = TfidfVectorizer(token_pattern=r"(?u)\b\w+\b", norm=norm, smooth_idf=smooth_idf, sublinear_tf=sublinear_tf, use_idf=use_idf)

    # Learn the vocabulary and weights from the fitting records only.
    vectorizer.fit(column.loc[feature_fit_rows])
    tfidf_matrix = vectorizer.transform(column)

    # Keep the source field identifiable when combining feature columns.
    column_prefix = f"{column.name}__" if column.name is not None else ""
    feature_columns = [f"{column_prefix}{token}" for token in vectorizer.get_feature_names_out()]

    return pd.DataFrame.sparse.from_spmatrix(tfidf_matrix, index=column.index, columns=feature_columns)

In [ ]:
from importlib.util import find_spec

if find_spec("sentence_transformers") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "sentence-transformers==5.1.2"])

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/LaBSE", tokenizer_kwargs={"do_lower_case": True}, similarity_fn_name="dot")
model.max_seq_length = 128
model.eval()

sentences = ["This is an example sentence", "Each sentence is converted"]
embeddings = model.encode(sentences, convert_to_numpy=True, show_progress_bar=False)

print("LaBSE loaded.")
print(model)
print(embeddings)


def transform_column_with_labse(column, alpha=0.5, feature_fit_rows=None):
    """
    Replace values containing Latin letters with their closest Hebrew match when the similarity exceeds alpha.
    """
    values = column.astype("string").str.strip().replace("", pd.NA)
    has_hebrew = values.str.contains(r"[א-ת]", na=False)
    has_latin = values.str.contains(r"[A-Za-z]", na=False)
    latin_cells = values.loc[has_latin]

    # Restrict Hebrew reference values to the fitting records when supplied.
    hebrew_values = values.where(has_hebrew & ~has_latin)
    if feature_fit_rows is not None:
        hebrew_values = hebrew_values.loc[feature_fit_rows]
    hebrew_names = sorted(hebrew_values.dropna().unique())

    transformed_column = column.copy()
    if latin_cells.empty or not hebrew_names:
        return transformed_column

    # Dot products between unit embeddings give cosine similarities.
    vectors = model.encode(latin_cells.tolist() + hebrew_names, batch_size=8, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)
    latin_count = len(latin_cells)
    similarities = vectors[:latin_count] @ vectors[latin_count:].T

    # Preserve the original value when its best match falls at or below the threshold.
    best_indices = similarities.argmax(axis=1)
    best_scores = similarities[np.arange(latin_count), best_indices]
    best_hebrew = pd.Series(np.asarray(hebrew_names)[best_indices], index=latin_cells.index, dtype="string")
    transformed_column.loc[latin_cells.index] = best_hebrew.where(best_scores > alpha, column.loc[latin_cells.index])

    return transformed_column# Cache the workbook bytes so rerunning the cell can reuse the download.

In [70]:
# Cache the workbook bytes so rerunning the cell can reuse the download.
if "raw" not in globals() or raw is None:
    URL = "https://raw.githubusercontent.com/shulruferor-sudo/Advance-Data-Science/main/Datasets/cbs_occupation_survey_sample/DUMMY_DATASET.xlsx"
    raw = urllib.request.urlopen(URL, timeout=90).read()

data = pd.read_excel(io.BytesIO(raw), sheet_name="Survey Dataset").dropna(axis=0, how="all").dropna(axis=1, how="all").iloc[:200]
metadata = pd.read_excel(io.BytesIO(raw), sheet_name="Metadata").dropna(axis=0, how="all").dropna(axis=1, how="any")

display(metadata.tail(5))
display(data.tail(2))

def resolve_left_references(data, value, columns):
    """
    Within each row, replace each marker with the nearest populated nonmarker value to its left.
    """
    selected = data.loc[:, columns]
    markers = selected.eq(value).fillna(False)
    blanks = selected.eq("").fillna(False)

    # Scan columns in the supplied order and carry the last usable value across each row.
    nearest_left = selected.mask(markers | blanks).ffill(axis=1)
    resolved = selected.mask(markers & nearest_left.notna(), nearest_left)

    return pd.concat([data.drop(columns=columns), resolved], axis=1).reindex(columns=data.columns)


def manual_explicits_transforms(value):
    """
    Apply the specified text replacements after removing the prefix and surrounding whitespace.
    """
    text_transformations = {"קרית גת": "קריית גת", "לא יכול לפרט": "חסוי", "4 עונות": "ארבע עונות", 'למ"ס': "הלשכה המרכזית לסטטיסטיקה"}
    if isinstance(value, str):
        value = value.removeprefix("א.א.").strip()
    return text_transformations.get(value, value)


def print_free_text_status(column):
    """
    Estimate whether the column is free text by comparing one-time and recurring values.
    """
    counts = column.value_counts()
    display(style_report_text(f"{column.name} is {'a' if counts.eq(1).sum() > counts.gt(1).sum() else 'not a'} free-text feature by this heuristic."))


def print_unique_settlements(column):
    """
    Display the number of distinct nonmissing values.
    """
    display(style_report_text(f"There are {column.nunique()} unique values in {column.name}."))


def print_mandatory_status(column):
    """
    Display whether the field appears mandatory based on observed missing values.
    """
    display(style_report_text(f"{column.name} appears {'optional' if column.isna().any() else 'mandatory'} based on the recorded values."))


def print_workplace_name_counts(column):
    """
    Display counts of missing entries, nonmissing entries, and distinct values.
    """
    display(style_report_text(f"Values are missing from {column.isna().sum():,} records and present in {column.count():,} records, with {column.nunique():,} distinct values overall."))


def group_values_by_script(column):
    """
    Group nonmissing values by the presence of Hebrew and ASCII Latin letters.
    """
    values = column.astype("string")
    has_hebrew = values.str.contains(r"[א-ת]", na=False)
    has_latin = values.str.contains(r"[A-Za-z]", na=False)

    return {"HEBREW ONLY": column.loc[has_hebrew & ~has_latin], "LATIN ONLY": column.loc[has_latin & ~has_hebrew], "MIXED HEBREW + LATIN": column.loc[has_hebrew & has_latin], "NO HEBREW OR LATIN LETTERS": column.loc[column.notna() & ~has_hebrew & ~has_latin]}


def print_script_frequencies(column, key):
    """
    Display value frequencies by script and list the record IDs matched by row index.
    """
    formatted_keys = key.astype("Int64").astype("string").fillna("—")

    for heading, values in group_values_by_script(column).items():
        record_count = len(values)
        value_count = values.nunique()
        section_summary = f"This section contains {record_count:,} {'record' if record_count == 1 else 'records'} and {value_count:,} distinct {'value' if value_count == 1 else 'values'}."
        display(style_report_text(heading, role="section"))
        display(style_report_text(section_summary, role="summary"))

        if values.empty:
            display(style_report_text("No records found.", role="empty"))
            continue

        frequencies = values.value_counts().rename_axis("Value").to_frame("Count")
        frequencies[key.name] = formatted_keys.loc[values.index].groupby(values, sort=False, observed=True).agg(", ".join)
        display(custom_style(frequencies.reset_index(), grouped_id_column=key.name))


for column_name in metadata.loc[metadata["פורמט"].eq("TEXT") & ~metadata["הסבר"].str.contains("סמל", regex=False, na=False), "משתנה"]:
    column = data[column_name]
    display(style_report_text(column_name, role="field"))
    data = resolve_left_references(data, 'כנ"ל', ["YeshuvAvoda", "ShemAvoda", "SugAvoda", "ShemMachlaka", "SugMachlaka", "EzoAvoda", "TeurPeula", "TeurTafkid"])
    print_free_text_status(column)
    print_unique_settlements(column)
    print_mandatory_status(column)
    print_workplace_name_counts(column)
    print_script_frequencies(column, data["ID"])


data = data.map(manual_explicits_transforms)

run_text_field_checks(data, metadata)

for column_name in ["ShemAvoda", "SugMachlaka", "EzoAvoda", "TeurPeula"]:
    data = pd.concat([data.drop(columns=column_name), transform_column_with_labse(data[column_name], feature_fit_rows=report_fit_rows.reindex(data.index, fill_value=False))], axis=1).reindex(columns=data.columns)

for column_name in ["ShemAvoda", "SugAvoda", "ShemMachlaka", "SugMachlaka", "EzoAvoda", "TeurPeula", "TeurTafkid"]:
    data = pd.concat([data.drop(columns=column_name), transform_to_tfidf(data[column_name], report_fit_rows.reindex(data.index, fill_value=False))], axis=1)

print("Final data shape shape:", data.shape)



,משתנה,הסבר,פורמט
13,shnotlimud,שנות הלימוד של הנסקר,INTEGER
14,Gil,גיל הנסקר,INTEGER
15,SemelAnafSofi,סמל ענף כלכלי של מקום העבודה - מורכב מ-4 ספרות לפי תקן בינלאומי,TEXT
16,TarichHathala,תאריך התחלת התפקיד,DATE
17,SemelMishlachSofi,סמל משלח יד - מורכב מ-4 ספרות לפי תקן בינלאומי,TEXT


,ID,YeshuvAvoda,ShemAvoda,SugAvoda,ShemMachlaka,SugMachlaka,EzoAvoda,TeurPeula,TeurTafkid,MenahelEtMi,MaamadAvoda,MakorSachar,TeudaGvoha,shnotlimud,Gil,TarichHathala,SemelAnafSofi,SemelMishlachSofi
198,199,חדרה,גן חדרה,גננת,חינוך,גננת,גננת,nan,nan,nan,1.000000,1.000000,4.000000,15.000000,38.000000,2011-02-01 00:00:00,8501,2342
199,200,ירושלים,"למ""ס",סטטיסטיקה,nan,nan,סימול סקרים,קידוד ענף כלכלי ומשלח יד,nan,nan,1.000000,1.000000,5.000000,15.000000,30.000000,2023-02-01 00:00:00,8411,4413


Value,Count,ID
תל אביב,35,"4, 5, 15, 20, 22, 29, 36, 47, 48, 52, 54, 61, 145, 146, 147, 148, 149, 150, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 164, 173, 175, 186, 196, 197"
ירושלים,21,"1, 2, 13, 18, 19, 25, 31, 32, 33, 35, 44, 45, 50, 51, 57, 58, 63, 64, 65, 96, 200"
חיפה,19,"6, 8, 37, 39, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 165, 180, 181, 191, 192"
רעננה,13,"12, 43, 137, 138, 139, 140, 141, 142, 143, 144, 171, 185, 193"
הרצליה,13,"10, 24, 41, 56, 109, 110, 111, 112, 113, 169, 177, 183, 189"
אילת,13,"28, 60, 74, 75, 76, 79, 80, 81, 82, 83, 179, 190, 195"
באר שבע,11,"3, 94, 95, 97, 98, 99, 100, 103, 163, 176, 188"
חדרה,11,"11, 40, 42, 114, 115, 117, 118, 170, 182, 184, 199"
אשקלון,11,"9, 85, 86, 87, 88, 89, 90, 91, 92, 93, 168"
אופקים,10,"26, 66, 67, 68, 69, 70, 71, 72, 73, 178"


Value,Count,ID
מסעדה,10,"13, 28, 44, 45, 60, 76, 91, 107, 108, 123"
משרד החינוך,9,"4, 35, 67, 98, 127, 132, 153, 161, 174"
בנק לאומי,7,"11, 42, 74, 105, 173, 185, 189"
משטרה,7,"6, 37, 69, 100, 131, 191, 196"
בית ספר תיכון,7,"26, 58, 89, 121, 134, 147, 156"
סופר פארם,6,"16, 48, 79, 111, 175, 182"
אינטל,6,"7, 8, 39, 70, 71, 102"
קניון עזריאלי,6,"29, 61, 92, 124, 150, 159"
קוסמטיקאית עצמאית,5,"17, 49, 80, 112, 176"
"למ""ס",5,"1, 32, 64, 95, 200"


Value,Count,ID
NVIDIA,4,"15, 47, 78, 110"
EMILIO,3,"10, 73, 104"
INTEL,2,"38, 101"
COGNYTE,2,"56, 119"
MICROSOFT,1,143
KPMG,1,145


Value,Count,ID
יועצת IT עצמאית,1,140


... (29 outputs left out)

Value,Count,ID
כלכלן,6,"12, 42, 43, 75, 105, 106"
טכנאית רדיותרפיה,4,"2, 33, 65, 96"
סגן מנהל אגף,3,"1, 32, 64"
טכנאי מחשבים,3,"11, 74, 169"
סוקר,1,95
פיתוח תכנה,1,137
מנהל פיתןח של 2 מוצרים,1,138
מנהלת משאבי אנוש של עובדיםן בישראל,1,139
"סמנכ""ל שיווק",1,143
תמיכה במחשוב,1,144


Value,Count,ID
בונה מודלים של AI,1,56
יועצת IT,1,140


Value,Count,ID
0,1,39


Final data shape shape: (200, 648)


In [ ]:
# Use the completed table as the source for model inputs.
from scipy import sparse
from sklearn.preprocessing import OneHotEncoder

record_ids = data["ID"].dropna().astype("int64")
occupation_codes = data.loc[record_ids.index, "SemelMishlachSofi"].astype("string").str.strip().str.replace(r"\.0$", "", regex=True)
occupation_codes = occupation_codes.map(lambda code: code.zfill(4) if code.isdecimal() else code, na_action="ignore")
complete_target = occupation_codes.str.fullmatch(r"[0-9]{4}", na=False)
labelled_targets = occupation_codes.loc[complete_target]
partial_targets = occupation_codes.loc[~complete_target]

model_features = data.loc[record_ids.index].drop(columns=["ID", "SemelAnafSofi", "SemelMishlachSofi"])

# Group records by the text features already produced.
text_profiles = model_features.filter(like="__").sparse.to_coo().tocsr()
text_profiles.sort_indices()
profile_keys = [(tuple(text_profiles.indices[start:stop]), tuple(text_profiles.data[start:stop])) for start, stop in zip(text_profiles.indptr[:-1], text_profiles.indptr[1:])]
profile_groups = pd.Series(pd.factorize(pd.Series(profile_keys, dtype=object))[0], index=model_features.index)

# Encode the remaining nominal fields for the classifiers.
categorical_fields = {"YeshuvAvoda": ("", None), "MaamadAvoda": (0, range(8)), "MakorSachar": (4, range(1, 5)), "TeudaGvoha": (9, range(1, 10))}

for column_name, (missing_value, codes) in categorical_fields.items():
    column = model_features[column_name].fillna(missing_value)
    column = column.astype("Int64").astype("string") if codes is not None else column.astype("string")
    values = column.to_frame()
    categories = [[str(code) for code in codes]] if codes is not None else "auto"
    encoder = OneHotEncoder(categories=categories, handle_unknown="ignore").fit(values.loc[labelled_targets.index])
    encoded = pd.DataFrame.sparse.from_spmatrix(encoder.transform(values), index=values.index, columns=encoder.get_feature_names_out())
    model_features = pd.concat([model_features.drop(columns=column_name), encoded], axis=1)

# Apply the existing numerical rules to the model inputs.
model_features["MenahelEtMi"] = model_features["MenahelEtMi"].fillna(0).astype("int64")

for column_name, (minimum, maximum) in {"shnotlimud": (8, 80), "Gil": (18, 120)}.items():
    column = model_features[column_name]
    column = column.where(column.between(minimum, maximum))
    model_features[column_name] = column.fillna(column.loc[labelled_targets.index].median())

reference_date = pd.Timestamp.now(tz="UTC").normalize().tz_localize(None)
model_features["TarichHathala"] = (reference_date - pd.to_datetime(model_features["TarichHathala"])).dt.days / 365.25

feature_names = model_features.columns.to_numpy()
training_matrix = model_features.loc[labelled_targets.index].astype(pd.SparseDtype("float64", 0)).sparse.to_coo().tocsr()
prediction_matrix = model_features.loc[partial_targets.index].astype(pd.SparseDtype("float64", 0)).sparse.to_coo().tocsr()

print(f"Complete labels: {len(labelled_targets)} | Incomplete labels: {len(partial_targets)} | Model features: {len(feature_names)}")


In [ ]:
# Keep identical encoded text profiles in the same split.
from sklearn.model_selection import GroupShuffleSplit

labelled_groups = profile_groups.loc[labelled_targets.index]
development_positions, test_positions = next(GroupShuffleSplit(n_splits=1, test_size=0.22, random_state=SEED).split(training_matrix, labelled_targets, labelled_groups))
development_matrix = training_matrix[development_positions]
test_matrix = training_matrix[test_positions]
development_targets = labelled_targets.iloc[development_positions]
test_targets = labelled_targets.iloc[test_positions]
development_groups = labelled_groups.iloc[development_positions]
validation_splits = list(GroupKFold(n_splits=3, shuffle=True, random_state=SEED).split(development_matrix, development_targets, development_groups))

results_directory = Path("classification_results")
results_directory.mkdir(exist_ok=True)

display(pd.DataFrame({"Records": [len(development_targets), len(test_targets), len(partial_targets)], "Complete classes": [development_targets.nunique(), test_targets.nunique(), 0]}, index=["Development", "Held-out test", "Incomplete labels"]))


In [ ]:
from itertools import product

import joblib
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.model_selection import GridSearchCV, ParameterGrid
from sklearn.svm import SVC

for package in ["xgboost", "lightgbm"]:
    if find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", package])

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

def build_ensemble():
    return VotingClassifier(estimators=[("logistic", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)), ("forest", RandomForestClassifier(n_estimators=80, class_weight="balanced", random_state=SEED, n_jobs=1)), ("xgboost", XGBClassifier(n_estimators=40, max_depth=2, learning_rate=0.1, eval_metric="mlogloss", tree_method="hist", random_state=SEED, n_jobs=1))], voting="soft", n_jobs=1)

model_families = {
    "Linear SVC": (LinearSVC(class_weight="balanced", random_state=SEED), {"classifier__C": [0.1, 1]}),

    "Logistic regression": (LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED), {"classifier__C": [0.1, 1]}),

    "RBF SVC": (SVC(C=1, class_weight="balanced", random_state=SEED), {}),

    "Complement NB": (ComplementNB(alpha=1), {}),

    "XGBoost": (VotingClassifier(estimators=[("xgboost", XGBClassifier(n_estimators=40, max_depth=2, learning_rate=0.1, eval_metric="mlogloss", tree_method="hist", random_state=SEED, n_jobs=1))], voting="soft"), {}),

    "LightGBM": (LGBMClassifier(n_estimators=40, num_leaves=7, min_child_samples=2, learning_rate=0.1, verbosity=-1, random_state=SEED, n_jobs=1), {}),

    "Random forest": (RandomForestClassifier(n_estimators=80, class_weight="balanced", random_state=SEED, n_jobs=1), {"classifier__min_samples_leaf": [1, 2]}),

    "Soft voting": (build_ensemble(), {})
}

class DigitExclusiveSelector(TransformerMixin, BaseEstimator):

    def __init__(self, min_support=3):
        self.min_support = min_support

    def fit(self, X, y):
        values = sparse.csc_matrix(X)
        targets = np.asarray(y, dtype=str)
        digit_codes = np.asarray([[code[position] for position in range(4)] for code in targets])

        self.support_ = np.diff(values.indptr)
        self.exclusive_ = np.zeros(values.shape[1], dtype=bool)

        for column in np.flatnonzero(self.support_ >= self.min_support):
            rows = values.indices[values.indptr[column]:values.indptr[column + 1]]
            self.exclusive_[column] = any(np.unique(digit_codes[rows, position]).size == 1 for position in range(4))

        self.mask_ = self.exclusive_.copy()

        if not self.mask_.any():
            self.mask_[:] = True

        return self

    def transform(self, X):

        return X[:, self.mask_]

dataset_options = pd.DataFrame([{"Dataset": f"D{i:02d}", "L2": l2, "Digit selection": select, "SVD": svd} for i, (l2, select, svd) in enumerate(product([False, True], repeat=3), start=1)])


# Fit each representation and classifier within the same grouped folds.
parameter_grids = []
candidate_descriptions = []

for dataset in dataset_options.to_dict("records"):
    for family, (classifier, parameters) in model_families.items():
        if dataset["SVD"] and family == "Complement NB":
            continue

        parameter_grids.append({"normalize": [Normalizer() if dataset["L2"] else "passthrough"], "select": [DigitExclusiveSelector(min_support=3) if dataset["Digit selection"] else "passthrough"], "svd": [TruncatedSVD(n_components=16, random_state=SEED) if dataset["SVD"] else "passthrough"], "classifier": [classifier], **parameters})
        candidate_descriptions.extend([{"Dataset": dataset["Dataset"], "Classifier": family, "L2": dataset["L2"], "Digit selection": dataset["Digit selection"], "SVD": dataset["SVD"], "Parameters": json.dumps(combination)} for combination in ParameterGrid(parameters)])

model_pipeline = Pipeline([("normalize", "passthrough"), ("select", "passthrough"), ("svd", "passthrough"), ("classifier", LogisticRegression())])
search = GridSearchCV(model_pipeline, parameter_grids, cv=validation_splits, scoring={"macro_f1": "f1_macro", "accuracy": "accuracy"}, refit="macro_f1", n_jobs=3, error_score="raise")

with joblib.parallel_backend("threading"):
    search.fit(development_matrix, development_targets)

scores = pd.DataFrame(search.cv_results_)[["mean_test_macro_f1", "std_test_macro_f1", "mean_test_accuracy"]]
model_results = pd.concat([pd.DataFrame(candidate_descriptions), scores], axis=1)
model_results.to_csv(results_directory / "cross_validation_results.csv", index=False, encoding="utf-8-sig")
best_configuration = model_results.loc[search.best_index_]
ranked_models = model_results.sort_values("mean_test_macro_f1", ascending=False, kind="stable").drop_duplicates(["Dataset", "Classifier"])
ranked_models = ranked_models.rename(columns={"Classifier": "Model", "mean_test_macro_f1": "Macro F1", "std_test_macro_f1": "Fold SD", "mean_test_accuracy": "Accuracy"})

print(f"{len(dataset_options)} representations, {len(model_results)} configurations, {len(validation_splits)} grouped folds.")
print("Exploratory scores. The text features were prepared before the split.")
display(ranked_models.head(18))


In [ ]:
# Export the same figure as interactive HTML and a 4K PNG.
import os
import shutil
from importlib.metadata import version
from packaging.version import Version

plot_packages = {"plotly": "6.3.0", "kaleido": "1.0.0"}
missing_plot_packages = [f"{package}>={minimum}" for package, minimum in plot_packages.items() if find_spec(package) is None or Version(version(package)) < Version(minimum)]
if missing_plot_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing_plot_packages])

import plotly.express as px
import plotly.io as pio

chrome_path = os.environ.get("BROWSER_PATH") or next((path for name in ["chrome-headless-shell", "google-chrome", "chromium", "chromium-browser"] if (path := shutil.which(name))), None)
if not chrome_path:
    os.environ["BROWSER_PATH"] = str(pio.get_chrome())

plot_directory = Path("plots")
plot_directory.mkdir(exist_ok=True)

def save_chart(figure, name):
    """
    Display the figure and export HTML and a 3840 × 2160 PNG.
    """
    figure.show()
    figure.write_html(plot_directory / f"{name}.html")
    figure.write_image(plot_directory / f"{name}_4k.png", width=1920, height=1080, scale=2)

# Compare model performance and the selected configuration's folds.
plot_results = ranked_models.head(18).sort_values("Macro F1").copy()
plot_results["Variant"] = plot_results["Dataset"] + " | " + plot_results["Model"]
save_chart(px.bar(plot_results, x="Macro F1", y="Variant", color="Macro F1", hover_data=["Accuracy", "Fold SD"], title="Occupation classifier comparison"), "classifier_comparison")

fold_overview = pd.DataFrame([{"Fold": f"Fold {fold + 1}", "Training records": len(fit_rows), "Validation records": len(validation_rows), "Macro F1": search.cv_results_[f"split{fold}_test_macro_f1"][search.best_index_], "Accuracy": search.cv_results_[f"split{fold}_test_accuracy"][search.best_index_]} for fold, (fit_rows, validation_rows) in enumerate(validation_splits)])
fold_overview.to_csv(results_directory / "selected_fold_results.csv", index=False, encoding="utf-8-sig")
display(fold_overview)
save_chart(px.bar(fold_overview, x="Fold", y="Macro F1", hover_data=["Accuracy", "Training records", "Validation records"], title="Selected classifier by validation fold"), "selected_fold_comparison")


In [ ]:
# Evaluate the selected classifier on held-out records.
selected_pipeline = search.best_estimator_
test_predictions = selected_pipeline.predict(test_matrix)
baseline = DummyClassifier(strategy="most_frequent").fit(development_matrix, development_targets)
baseline_predictions = baseline.predict(test_matrix)

held_out_predictions = pd.DataFrame({"ID": record_ids.loc[test_targets.index], "Recorded occupation": test_targets, "Predicted occupation": test_predictions})
held_out_predictions["Correct"] = held_out_predictions["Recorded occupation"].eq(held_out_predictions["Predicted occupation"])
held_out_predictions.to_csv(results_directory / "held_out_predictions.csv", index=False, encoding="utf-8-sig")

held_out_metrics = pd.DataFrame([{"Model": name, "Accuracy": accuracy_score(test_targets, predictions), "Macro F1": f1_score(test_targets, predictions, average="macro", zero_division=0)} for name, predictions in [(best_configuration["Classifier"], test_predictions), ("Majority baseline", baseline_predictions)]])
prefix_metrics = pd.DataFrame([{"Digits": width, "Agreement": np.mean([prediction[:width] == actual[:width] for prediction, actual in zip(test_predictions, test_targets)])} for width in range(1, 5)])
held_out_metrics.to_csv(results_directory / "held_out_metrics.csv", index=False, encoding="utf-8-sig")
prefix_metrics.to_csv(results_directory / "hierarchical_agreement.csv", index=False, encoding="utf-8-sig")
display(held_out_metrics)
display(held_out_predictions)

metric_plot = held_out_metrics.melt(id_vars="Model", var_name="Metric", value_name="Score")
save_chart(px.bar(metric_plot, x="Model", y="Score", color="Metric", barmode="group", title="Held-out model and majority baseline"), "held_out_summary")
save_chart(px.line(prefix_metrics, x="Digits", y="Agreement", markers=True, title="Held-out agreement by occupation-code prefix"), "held_out_prefix")


In [ ]:
# Fit the selected configuration on all complete labels.
final_pipeline = clone(selected_pipeline).fit(training_matrix, labelled_targets)
predicted_codes = final_pipeline.predict(prediction_matrix)

if hasattr(final_pipeline, "predict_proba"):
    prediction_scores = final_pipeline.predict_proba(prediction_matrix)
else:
    decision_scores = np.asarray(final_pipeline.decision_function(prediction_matrix))
    prediction_scores = decision_scores if decision_scores.ndim == 2 else np.column_stack([-decision_scores, decision_scores])

# Rank compatible classes using the digits already recorded.
ranked_codes = final_pipeline.classes_[np.argsort(-prediction_scores, axis=1, kind="stable")]
compatible_predictions = [next((candidate for candidate in candidates if all(known == "X" or known == digit for known, digit in zip(recorded, candidate))), pd.NA) for recorded, candidates in zip(partial_targets, ranked_codes)]
prediction_results = pd.DataFrame({"ID": record_ids.loc[partial_targets.index], "Recorded occupation": partial_targets, "Model prediction": predicted_codes, "Digit-compatible prediction": compatible_predictions})

prediction_results.to_csv(results_directory / "incomplete_label_predictions.csv", index=False, encoding="utf-8-sig")
joblib.dump(final_pipeline, results_directory / "selected_pipeline.joblib")
pd.Series(feature_names, name="Feature").to_csv(results_directory / "selected_feature_names.csv", index=False, encoding="utf-8-sig")
display(prediction_results)
